In [1]:
import torch
import numpy as np

from torch import nn, optim

In [16]:
modelo = nn.Sequential(
    nn.Linear(2,1),
    nn.Sigmoid()
)

In [3]:
datos = [[0,1],[1,1],[0,0]]
x_datos = torch.tensor(datos, dtype=torch.float)
resultado = modelo(x_datos)

print(resultado)

tensor([[1.],
        [1.],
        [1.]], grad_fn=<SoftmaxBackward0>)


In [4]:
# Compuerta AND
datos = [[0,0], [0,1], [1,0], [1,1]]
salidas = [[0], [0], [0], [1]]

x_datos = torch.tensor(datos, dtype=torch.float)
y_salidas = torch.tensor(salidas, dtype=torch.float)

In [5]:
datos_prueba = [[0,0], [0,1], [0,0], [1,0], [1, 1]]
salidas_prueba = [[0], [0], [0], [0], [1]]
x_prueba = torch.tensor(datos_prueba, dtype=torch.float)
y_prueba = torch.tensor(salidas_prueba, dtype=torch.float)

In [ ]:
# Se crean objetos especiales de Pytorch para manejar los datos (entrenamiento y evaluación)
from torch.utils.data import TensorDataset, DataLoader

dataset_entrenamiento = TensorDataset(x_datos, y_salidas)
dataset_evaluacion = TensorDataset(x_prueba, y_prueba)

entrenamiento_loader = DataLoader(dataset=dataset_entrenamiento, batch_size=1, shuffle=True)
evaluacion_loader = DataLoader(dataset=dataset_evaluacion, batch_size=1, shuffle=True)

In [25]:
from tqdm import tqdm

loss_fn = nn.BCELoss()
optimizer = optim.SGD(modelo.parameters(), lr=0.001, momentum=0.9)

# Entrenamiento: Loop sobre el dataset múltiples veces (épocas)
for epoca in tqdm(range(1000)):
    modelo.train() # Bandera para poner el modelo en modo entrenamiento
    for i, datos in enumerate(entrenamiento_loader, 0):
        # Los datos son batches (lotes) [entradas etiquetas]
        entradas, salidas_esperadas = datos

        # Inicializar el gradiente
        optimizer.zero_grad()

        # Calcular la salida de la red (Feed Forward)
        salida_red = modelo(entradas)

        # Calcular el error y realizar la propagación del mismo hacia
        # atrás (Backprogation error)
        perdida = loss_fn(salida_red, salidas_esperadas)
        perdida.backward()

        # Actualizar pesos y leraning rate
        optimizer.step()


100%|██████████| 1000/1000 [00:01<00:00, 538.22it/s]


In [26]:
# Evaluar el modelo
correctos = 0 # Entradas clasificadas correctamente
total = 0

modelo.eval() # Bandera para poner al modelo en modo de evaluación
with torch.no_grad(): # RECOMENDADO: Le dice al backend de Pytorch que no se calcularán gradientes: + óptimo
    for datos in evaluacion_loader:
        entradas, salidas_esperadas = datos
        salida_red = modelo(entradas)
        salida_binaria = (salida_red > 0.5).float()

        # Comparación con resultados esperados
        print(f"Esperado {salidas_esperadas} Predicho {salida_binaria}")
        total += salidas_esperadas.size(0)
        correctos += (salida_binaria == salidas_esperadas).sum().item()


Esperado tensor([[0.]]) Predicho tensor([[0.]])
Esperado tensor([[0.]]) Predicho tensor([[0.]])
Esperado tensor([[0.]]) Predicho tensor([[0.]])
Esperado tensor([[0.]]) Predicho tensor([[0.]])
Esperado tensor([[1.]]) Predicho tensor([[1.]])


In [27]:
print(correctos/total)

1.0


## Ejercicio

Diseñar redes neuronales que puedan realizar las operaciones:
* OR
* XOR